In [0]:
-- Latest failed jobs
CREATE OR REPLACE VIEW data_governance.gold_analytics_views.gold_vw_failures_recent AS
select account_id,workspace_id,job_id,job_name,run_id,run_type,trigger_type,result_state,period_start_time as job_start_time,period_end_time as job_end_time,termination_type FROM
(select * , ROW_NUMBER() OVER (PARTITION BY job_id ORDER BY period_start_time DESC) AS rn FROM data_governance.gold_lineage.fact_job_performance where result_state!='SUCCEEDED')t
where rn<=5 ;

In [0]:
-- Queries with high execution time and multiple runs
CREATE OR REPLACE VIEW data_governance.gold_analytics_views.gold_vw_heavy_queries AS
WITH aggregated AS (
    SELECT
        statement_text,
        COUNT(*) AS execution_count,
        ROUND(AVG(query_duration_sec), 2) AS avg_duration_sec,
        MAX(query_duration_sec) AS max_duration_sec,
        SUM(read_mb) AS total_read_mb
    FROM data_governance.silver_lineage_analysis.lineage_query_history
    GROUP BY statement_text
    HAVING COUNT(*) > 1
),
ranked AS (
    SELECT *,
           ROW_NUMBER() OVER (ORDER BY avg_duration_sec DESC) AS rn
    FROM aggregated
),
sample_query AS (
    SELECT *
    FROM (
        SELECT *,
               ROW_NUMBER() OVER (PARTITION BY statement_text ORDER BY query_duration_sec DESC) AS rnk
        FROM data_governance.silver_lineage_analysis.lineage_query_history
    ) t
    WHERE rnk = 1
)
SELECT
    s.account_id,
    s.workspace_id,
    s.executed_by AS executor_id,
    s.execution_status,
    s.statement_id AS query_statement_id,
    s.statement_text,
    s.statement_type,
    r.execution_count,
    r.avg_duration_sec,
    r.max_duration_sec,
    r.total_read_mb
FROM ranked r
JOIN sample_query s
ON r.statement_text = s.statement_text
WHERE r.rn <= 10;

In [0]:
-- user resource usage.
CREATE OR REPLACE VIEW data_governance.gold_analytics_views.gold_vw_user_resource_usage AS
SELECT
    executed_by,
    SUM(read_mb) AS total_read_mb,
    SUM(written_mb) AS total_written_mb,
    SUM(shuffle_read_mb) AS total_shuffle_mb
FROM data_governance.silver_lineage_analysis.lineage_query_history
GROUP BY executed_by;

In [0]:
--
SELECT
    job_id,
    COUNT(*) AS total_queries,
    AVG(query_duration_sec) AS avg_duration_sec
FROM data_governance.silver_lineage_analysis.lineage_query_history
GROUP BY job_id;

In [0]:
-- daily query rund and usage details
CREATE OR REPLACE VIEW data_governance.gold_analytics_views.gold_vw_query_daily_stats AS
SELECT
    event_date,
    COUNT(*) AS total_queries,
    COUNT(CASE WHEN execution_status = 'FINISHED' THEN 1 END) AS success_count,
    COUNT(CASE WHEN execution_status != 'FINISHED' THEN 1 END) AS failure_count,
    ROUND(AVG(query_duration_sec), 2) AS avg_duration_sec,
    ROUND(SUM(read_mb), 2) AS total_read_mb,
    ROUND(SUM(written_mb), 2) AS total_written_mb

FROM data_governance.silver_lineage_analysis.lineage_query_history
GROUP BY event_date;

In [0]:
-- cost summary as per cluster usage
CREATE OR REPLACE VIEW data_governance.gold_analytics_views.gold_vw_cost_daily_summary AS
SELECT
    usage_date,

    -- user normalization
    COALESCE(user_id, created_by, resource_owner) AS user_name,

    sku_name,
    cloud,

    -- usage
    COUNT(*) AS total_usage_records,
    ROUND(SUM(usage_quantity), 2) AS total_usage_qty,
    ROUND(SUM(usage_duration_minutes), 2) AS total_duration_mins,

    -- 💰 cost
    ROUND(SUM(total_cost), 2) AS total_cost,

    -- 🔥 key insights
    ROUND(SUM(total_cost) / NULLIF(SUM(usage_quantity), 0), 4) AS cost_per_dbu,
    ROUND(SUM(total_cost) / NULLIF(SUM(usage_duration_minutes), 0), 6) AS cost_per_minute,
    ROUND(AVG(total_cost), 4) AS avg_cost_per_record,

    -- serverless split (important for your case)
    ROUND(SUM(CASE WHEN is_serverless THEN total_cost ELSE 0 END), 2) AS serverless_cost

FROM data_governance.gold_cost.fact_cost_usage

GROUP BY
    usage_date,
    COALESCE(user_id, created_by, resource_owner),
    sku_name,
    cloud

ORDER BY total_cost DESC;

In [0]:
--all job run details
CREATE OR REPLACE VIEW data_governance.gold_analytics_views.gold_vw_job_run_summary AS
SELECT
    fjp.job_id,

    -- job name (fallback if mismatch)
    COALESCE(djm.name, fjp.job_name) AS job_name,
    COALESCE(djm.run_as_user_name, djm.creator_user_name) AS job_owner,
    djm.job_category,

    COUNT(fjp.run_id) AS total_runs,

    SUM(CASE WHEN fjp.result_state = 'SUCCESS' THEN 1 ELSE 0 END) AS success_count,
    SUM(CASE WHEN fjp.result_state != 'SUCCESS' THEN 1 ELSE 0 END) AS failure_count,
    ROUND(
        SUM(CASE WHEN fjp.result_state = 'SUCCESS' THEN 1 ELSE 0 END) 
        / NULLIF(COUNT(fjp.run_id), 0),
        4
    ) AS success_rate,

    ROUND(AVG(fjp.run_duration_seconds), 2) AS avg_run_duration_sec,
    MAX(fjp.run_duration_seconds) AS max_run_duration_sec,

    -- latency
    ROUND(AVG(fjp.queue_duration_seconds), 2) AS avg_queue_time_sec,
    ROUND(AVG(fjp.setup_duration_seconds), 2) AS avg_setup_time_sec,
    ROUND(AVG(fjp.cleanup_duration_seconds), 2) AS avg_cleanup_time_sec,
    -- time
    MIN(fjp.period_start_time) AS first_run_time,
    MAX(fjp.period_end_time) AS last_run_time

FROM data_governance.gold_lineage.fact_job_performance fjp

LEFT JOIN data_governance.gold_lineage.dim_jobs_metadata djm
    ON fjp.job_id = djm.job_id

GROUP BY
    fjp.job_id,
    COALESCE(djm.name, fjp.job_name),
    COALESCE(djm.run_as_user_name, djm.creator_user_name),
    djm.job_category

ORDER BY total_runs DESC;


In [0]:
--users activity
CREATE OR REPLACE VIEW data_governance.gold_analytics_views.gold_vw_user_activity AS
WITH base_activity AS (
    SELECT
        ua.event_date,
        ua.workspace_id,
        ua.user_email,

        COUNT(DISTINCT ua.event_id) AS total_actions,
        COUNT(DISTINCT ua.service_name) AS services_used,
        COUNT(DISTINCT ua.action_name) AS unique_actions,

        SUM(CASE WHEN ua.is_system_user THEN 1 ELSE 0 END) AS system_actions,
        SUM(CASE WHEN NOT ua.is_system_user THEN 1 ELSE 0 END) AS user_actions,

        MAX(ua.event_timestamp) AS last_activity_time

    FROM data_governance.gold_access.fact_user_access_activity ua
    GROUP BY
        ua.event_date,
        ua.workspace_id,
        ua.user_email
),


ai_usage AS (
    SELECT
        au.event_date,
        au.workspace_id,
        au.user_id,

        COUNT(DISTINCT au.event_id) AS ai_assistant_uses,
        COUNT(DISTINCT au.session_id) AS ai_sessions

    FROM data_governance.gold_access.fact_assistant_uses au
    GROUP BY
        au.event_date,
        au.workspace_id,
        au.user_id
),


permissions AS (
    SELECT
        dtp.grantee AS user_email,
        COUNT(DISTINCT dtp.full_table_path) AS tables_accessible
    FROM data_governance.gold_access.dim_table_permissions dtp
    GROUP BY dtp.grantee
)


SELECT
    ba.event_date,
    ba.workspace_id,
    ba.user_email AS user_id,
    ba.total_actions,
    ba.services_used,
    ba.unique_actions,
    ba.system_actions,
    ba.user_actions,

    COALESCE(ai.ai_assistant_uses, 0) AS ai_assistant_uses,
    COALESCE(ai.ai_sessions, 0) AS ai_sessions,

    COALESCE(p.tables_accessible, 0) AS tables_accessible,

    ba.last_activity_time

FROM base_activity ba

LEFT JOIN ai_usage ai
    ON ba.workspace_id = ai.workspace_id
    AND ba.event_date = ai.event_date
    AND ba.user_email = ai.user_id

LEFT JOIN permissions p
    ON ba.user_email = p.user_email

ORDER BY ba.total_actions DESC;

In [0]:
--inactive users from a month

CREATE OR REPLACE VIEW data_governance.gold_analytics_views.gold_vw_inactive_users AS
WITH last_activity AS (
    SELECT
        ua.workspace_id,
        ua.user_email AS user_id,
        MAX(ua.event_timestamp) AS last_activity_time
    FROM data_governance.gold_access.fact_user_access_activity ua
    GROUP BY
        ua.workspace_id,
        ua.user_email
)

SELECT
    la.workspace_id,
    la.user_id,
    la.last_activity_time,

    -- 🔥 inactivity calculation
    DATEDIFF(current_date, DATE(la.last_activity_time)) AS days_inactive

FROM last_activity la

-- ✅ inactive users (30 days)
WHERE la.last_activity_time < current_date - INTERVAL 30 DAYS

-- optional cleanup
AND la.user_id NOT IN ('System-User', 'unknown', '', 'System user')

ORDER BY days_inactive DESC;

In [0]:
--tables accesses
CREATE OR REPLACE VIEW data_governance.gold_analytics_views.gold_vw_access_audit AS
SELECT
    ua.event_date,
    ua.event_timestamp,

    -- workspace context
    ua.workspace_id,

    -- 👤 who
    ua.user_email AS user_id,
    ua.is_system_user,

    -- ⚙️ action
    ua.service_name,
    ua.action_name,

    -- 📊 what (table info)
    ua.catalog_name,
    dtp.schema_name,
    dtp.table_name,
    dtp.full_table_path,

    -- 🔐 permission context
    dtp.permission_category,
    dtp.is_grantable_flag,

    -- 🧾 metadata
    ua.event_id

FROM data_governance.gold_access.fact_user_access_activity ua

LEFT JOIN data_governance.gold_access.dim_table_permissions dtp
    ON ua.user_email = dtp.grantee
    AND ua.catalog_name = dtp.catalog_name

-- ✅ filter only table-related activity (important)
WHERE ua.catalog_name IS NOT NULL

-- optional cleanup
AND ua.user_email NOT IN ('System-User', 'unknown', '', 'System user')

ORDER BY ua.event_timestamp DESC;

In [0]:
-- access and usage by tables (need to be corrected once table is added in system)
CREATE OR REPLACE VIEW data_governance.gold_analytics_views.gold_vw_table_usage AS
SELECT
    account_id,
    workspace_id,

    metastore_name,
    catalog_name,
    schema_name,
    table_name,

    COUNT(*) AS total_operations,
    COUNT(DISTINCT operation_id) AS unique_operations,

    MIN(start_time) AS first_used_at,
    MAX(end_time) AS last_used_at,

    DATEDIFF(MAX(end_time), MIN(start_time)) AS active_days

FROM system.storage.predictive_optimization_operations_history

GROUP BY
    account_id,
    workspace_id,
    metastore_name,
    catalog_name,
    schema_name,
    table_name

ORDER BY
    total_operations DESC;

In [0]:
-- queries and codes which high load
CREATE OR REPLACE VIEW data_governance.gold_analytics_views.gold_vw_large_scans AS
SELECT
    account_id,
    workspace_id,

    statement_text,

    COUNT(*) AS execution_count,

    ROUND(SUM(read_bytes) / (1024 * 1024 * 1024), 2) AS total_read_gb,
    ROUND(AVG(read_bytes) / (1024 * 1024 * 1024), 2) AS avg_read_gb,
    ROUND(MAX(read_bytes) / (1024 * 1024 * 1024), 2) AS max_read_gb,

    ROUND(SUM(read_rows), 0) AS total_rows_read,

    MIN(start_time) AS first_scan,
    MAX(end_time) AS last_scan

FROM data_governance.gold_lineage.fact_query_info

WHERE read_bytes IS NOT NULL
  AND read_bytes > 0
  AND statement_type = 'select'   -- focus on scans

GROUP BY
    account_id,
    workspace_id,
    statement_text

HAVING SUM(read_bytes) > 10 * 1024 * 1024   -- > 1 GB threshold
;